# Leaflet cluster map of talk and poster locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` and `_posters/` directories contain `.md` files of all your talks and posters. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and generates an interactive map with color-coded markers (red for talks, blue for posters).

In [1]:
# Start by installing the dependencies
!pip install python-frontmatter --upgrade
import frontmatter
import glob
import json
import time
from datetime import date
from geopy import Nominatim
from geopy.exc import GeopyError


In [2]:
# Collect the Markdown files from both talks and posters
g = glob.glob("_talks/*.md") + glob.glob("_posters/*.md")

In [3]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="academicpages.github.io")
location_data = []  # List of dicts: {description, lat, lon, type, color}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [4]:
# Perform geolocation
def is_virtual_location(value):
    if not value:
        return False
    lowered = value.lower()
    return "visioconf" in lowered or "visio" in lowered or "online" in lowered or "zoom" in lowered

def geocode_with_fallback(primary, fallback):
    for query in (primary, fallback):
        if not query:
            continue
        try:
            result = geocoder.geocode(query, timeout=TIMEOUT)
        except (GeopyError, ValueError) as ex:
            print(f"Error: geocode failed on input {query} with message {ex}")
            result = None
        time.sleep(1)
        if result:
            return result, query
    return None, None

for file in g:
    # Read the file
    data = frontmatter.load(file)
    data = data.to_dict()

    # Only show items explicitly enabled for the map
    if not data.get('showonmap', False):
        continue

    # Determine type (talk or poster) and color
    item_type = data.get('collection', 'talk')  # 'talks' or 'posters'
    item_date = data.get('date', None)
    if hasattr(item_date, 'date'):
        item_date = item_date.date()
    elif isinstance(item_date, str):
        item_date = date.fromisoformat(item_date)
    upcoming = item_date is not None and item_date > date.today()
    if 'poster' in item_type:
        item_type = 'poster'
        color = '#00A896' if upcoming else '#2E5CA6'  # Cyan for upcoming, blue for past
    else:
        item_type = 'talk'
        color = '#FF8C00' if upcoming else '#D62828'  # Orange for upcoming, red for past

    # Prepare the description (display can differ from geocoding input)
    title = data.get('title', '').strip()
    location = data.get('location', '').strip()
    display_location = data.get('location_display', '').strip()
    if is_virtual_location(location):
        location = ""
    if not display_location:
        display_location = location
    if display_location:
        description = f"{title}<br />{display_location}"
    else:
        description = title

    # Geocode the location and report the status
    result, used_query = geocode_with_fallback(location, location)
    if not result:
        print(f"Warning: geocode returned no result for {description}")
        continue
    
    # Get permalink for link to full page
    permalink = data.get('permalink', '')

    location_data.append({
        'description': description,
        'latitude': result.latitude,
        'longitude': result.longitude,
        'type': item_type,
        'upcoming': upcoming,
        'color': color,
        'permalink': permalink
    })
    print(description, f"(geocoded from: {used_query})", f"[{item_type}]", result)

Combining Finite Element Methods and Neural Networks to Solve Elliptic Problems on 2D Geometries<br />ENSAM, Paris, France (geocoded from: ENSAM, Paris, France) [talk] École Nationale Supérieure des Arts et Métiers - HESAM Université, Cour Pinel, Quartier de la Salpêtrière, Paris 13e Arrondissement, Paris, Île-de-France, France métropolitaine, 75013, France


Enriching continuous Lagrange finite element approximation spaces using neural networks<br />Saint-Jacut-de-la-Mer, France (geocoded from: Saint-Jacut-de-la-Mer, France) [talk] Saint-Jacut-de-la-Mer, Dinan, Côtes-d'Armor, Bretagne, France métropolitaine, 22750, France


Enriching continuous Lagrange finite element approximation spaces using neural networks<br />Institut Montpelliérain Alexander Grothendieck (IMAG), Montpellier, France (geocoded from: Faculté des Sciences de Montpellier, Montpellier, France) [talk] Faculté des sciences de Montpellier (campus Triolet), Place Eugène Bataillon, Hôpitaux-Facultés, Montpellier, Hérault, Occitanie, France métropolitaine, 34095, France


Enriching continuous Lagrange finite element approximation spaces using neural networks<br />Edinburgh, United Kingdom (geocoded from: Edinburgh, United Kingdom) [talk] City of Edinburgh, Alba / Scotland, United Kingdom


Enriching continuous Lagrange finite element approximation spaces using neural networks<br />McGill University, Montreal, Canada (geocoded from: McGill University, Montreal, Canada) [talk] McGill University, 845, Rue Sherbrooke Ouest, Ville-Marie, Montréal, Agglomération de Montréal, Montréal (région administrative), Québec, H3A 3P8, Canada


Combining Finite Element methods and Neural Networks to solve elliptic problem on complex 2D geometries<br />École Normale Supérieure de Lyon, Lyon, France (geocoded from: École Normale Supérieure de Lyon, Lyon, France) [poster] École Normale Supérieure de Lyon - Site Jacques Monod, Allée d'Italie, Le Bon Lait, Gerland, Lyon 7e Arrondissement, Lyon, Métropole de Lyon, Rhône, Auvergne-Rhône-Alpes, France métropolitaine, 69007, France


Combining Finite Element methods and Neural Networks to solve elliptic problem on complex 2D geometries<br />UFR, Strasbourg, France (geocoded from: UFR, Strasbourg, France) [poster] Pôle ingénierie) de l'UFR de Physique et Ingénierie, Rue du Maréchal Lefèbvre, Zone d'Activités de la Plaine des Bouchers, Meinau, Strasbourg, Bas-Rhin, Collectivité européenne d'Alsace, Grand Est, France métropolitaine, 67100, France


Enriching continuous Lagrange finite element approximation spaces using neural networks<br />Politecnico di Milano, Milan, Italy (geocoded from: Politecnico di Milano, Milan, Italy) [poster] Politecnico di Milano, Sede Milano Città Studi, Via Giovanni Villani, Vallazze, Città Studi, Municipio 3, Milano, Rodano, Milano, Lombardia, 20133, Italia


In [5]:
# Save the map data as org-locations.js
import os
os.makedirs('talkmap', exist_ok=True)

with open('talkmap/org-locations.js', 'w', encoding='utf-8') as f:
    f.write('var addressPoints = [\n')
    for i, item in enumerate(location_data):
        comma = ',' if i < len(location_data) - 1 else ''
        f.write(f'  {{\n')
        f.write(f'    "description": "{item["description"]}",\n')
        f.write(f'    "latitude": {item["latitude"]},\n')
        f.write(f'    "longitude": {item["longitude"]},\n')
        f.write(f'    "type": "{item["type"]}",\n')
        f.write(f'    "color": "{item["color"]}",\n')
        f.write(f'    "permalink": "{item["permalink"]}"\n')
        f.write(f'  }}{comma}\n')
    f.write('];\n')

print(f"\nGenerated talkmap/org-locations.js with {len(location_data)} points")
print(f"  - Talks (red): {sum(1 for x in location_data if x['type'] == 'talk')}")
print(f"  - Posters (blue): {sum(1 for x in location_data if x['type'] == 'poster')}")


Generated talkmap/org-locations.js with 8 points
  - Talks (red): 5
  - Posters (blue): 3
